# Beige Book Pipeline v2 (SDK-only)

This notebook walks through a full example of building forecasting training data from Beige Book reports, then fine-tuning and evaluating a model.

If this is your first time with this workflow, read top-to-bottom and run cells in order. By the end, you will have:
- a FileSet of source PDFs,
- a generated and labeled dataset,
- a train/test split,
- a fine-tuned model job,
- and evaluation metrics to compare against the base model.

**Pipeline stages:**
1. Download Beige Book PDFs
2. Upload PDFs into a FileSet with date metadata
3. Build and run one `QuestionPipeline`:
   - `FileSetSeedGenerator` to create seed excerpts
   - `ForwardLookingQuestionGenerator` to propose forecast questions
   - `FileSetDocumentLabeler(TemporalConstraint.NEXT_DOCUMENT)` to label using later reports
   - `QdrantContextGenerator(temporal_direction="before")` to attach earlier context only
   - `QuestionRenderer(template)` to format final training prompts
4. Prepare training data with `prepare_for_training`
5. Launch training with `lr.training.create`
6. Run evaluation with `lr.evals.run`
7. Review results

In [1]:
import os
import requests
from pathlib import Path

from lightningrod import (
    LightningRod,
    BinaryAnswerType,
    QdrantContextGenerator,
    FileSetSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    TemporalConstraint,
)
from lightningrod._generated.models import (
    FileSetMetadataSchemaInput,
    MetadataFieldDefinitionInput,
    MetadataFieldType,
)

lr = LightningRod(
    api_key=os.getenv("LR_PROD_API_KEY"),
    base_url="https://api.lightningrod.ai/api/public/v1",
)

## 1. Download Beige Book PDFs

Start by downloading the raw source documents. This gives you a local set of Beige Book files (`files/`) that the rest of the notebook uses.

In [2]:
BEIGE_BOOK_DATES = [
    "20230118", "20230308", "20230419", "20230531", "20230712", "20230906", "20231018", "20231129",
    "20240117", "20240306", "20240417", "20240529", "20240717", "20240904", "20241023", "20241204",
    "20250115", "20250305", "20250423", "20250604", "20250716", "20250903", "20251015", "20251126",
    "20260114", "20260304",
]

BASE_URL = "https://www.federalreserve.gov/monetarypolicy/files/BeigeBook_{date}.pdf"
pdf_dir = Path("files")
pdf_dir.mkdir(exist_ok=True)

for date_str in BEIGE_BOOK_DATES:
    out_path = pdf_dir / f"BeigeBook_{date_str}.pdf"
    if out_path.exists():
        continue
    r = requests.get(BASE_URL.format(date=date_str))
    r.raise_for_status()
    out_path.write_bytes(r.content)
    print(f"downloaded: {out_path.name}")

print(f"{len(list(pdf_dir.glob('BeigeBook_*.pdf')))} PDFs ready")

downloaded: BeigeBook_20230118.pdf
downloaded: BeigeBook_20230308.pdf
downloaded: BeigeBook_20230419.pdf
downloaded: BeigeBook_20230531.pdf
downloaded: BeigeBook_20230712.pdf
downloaded: BeigeBook_20230906.pdf
downloaded: BeigeBook_20231018.pdf
downloaded: BeigeBook_20231129.pdf
downloaded: BeigeBook_20240117.pdf
downloaded: BeigeBook_20240306.pdf
downloaded: BeigeBook_20240417.pdf
downloaded: BeigeBook_20240529.pdf
downloaded: BeigeBook_20240717.pdf
downloaded: BeigeBook_20240904.pdf
downloaded: BeigeBook_20241023.pdf
downloaded: BeigeBook_20241204.pdf
downloaded: BeigeBook_20250115.pdf
downloaded: BeigeBook_20250305.pdf
downloaded: BeigeBook_20250423.pdf
downloaded: BeigeBook_20250604.pdf
downloaded: BeigeBook_20250716.pdf
downloaded: BeigeBook_20250903.pdf
downloaded: BeigeBook_20251015.pdf
downloaded: BeigeBook_20251126.pdf
downloaded: BeigeBook_20260114.pdf
downloaded: BeigeBook_20260304.pdf
26 PDFs ready


## 2. FileSet setup

Create a FileSet and upload the downloaded PDFs so the SDK can query them during generation and labeling.

Each file gets `date` and `file_date` metadata. The `file_date` value is important because later steps use it to enforce time order (future docs for labels, past docs for context).

If you already have a populated FileSet, skip creation/upload and set `FILESET_ID` directly.

In [3]:
schema = FileSetMetadataSchemaInput(fields=[
    MetadataFieldDefinitionInput(
        name="date", field_type=MetadataFieldType.STRING, required=True,
    ),
])
fileset = lr.filesets.create(
    name="Beige Book Reports",
    description="Federal Reserve Beige Book PDFs 2023-2026",
    metadata_schema=schema,
)
FILESET_ID = fileset.id
print(f"Created FileSet: {FILESET_ID}")

from datetime import datetime, timezone
pre_file_paths = sorted(pdf_dir.glob("BeigeBook_*.pdf"))
pre_metadata = {}
for p in pre_file_paths:
    date_str = p.stem.replace("BeigeBook_", "")
    pre_metadata[p.name] = {
        "date": date_str,
        "file_date": datetime.strptime(date_str, "%Y%m%d").replace(tzinfo=timezone.utc),
    }
result = lr.filesets.upload_files(
    FILESET_ID,
    pre_file_paths,
    metadata=pre_metadata,
)

Created FileSet: e902b5ea-843a-4585-b59d-cdafd70ca715


## 3. Build pipeline

Define the end-to-end `QuestionPipeline` in one place. This pipeline controls how examples are created:
- what text becomes a seed,
- what questions are asked,
- how answers are labeled,
- what context is attached,
- and how the final prompt is rendered.

In [4]:
from lightningrod._generated.models import FileSetDocumentLabeler


answer_type = BinaryAnswerType()

template = (
    "You are an expert economic forecaster analyzing Federal Reserve Beige Book reports. "
    "You will be given a question about a future economic outcome, a prior Beige Book excerpt "
    "(context only — the labeled outcome is determined by a later report), and historical context "
    "from past Beige Book reports. Predict the probability that the outcome will occur.\n\n"
    "TODAY'S DATE: {question_date}\n\n"
    "QUESTION:\n{question_text}\n\n"
    "RESOLUTION CRITERIA:\n{resolution_criteria}\n\n"
    "PRIOR REPORT EXCERPT (context only — answer comes from a future report):\n{seed_text}\n\n"
    "HISTORICAL CONTEXT (past Beige Book excerpts):\n{context}\n\n"
    "Think step by step, then output your prediction.\n\n"
    "ANSWER FORMAT:\n{answer_instructions}"
)

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=FILESET_ID,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        questions_per_seed=5,
        answer_type=answer_type,
        instructions=(
            "Generate questions about whether specific economic outcomes will occur in the near future "
            "(decrease, increase, slow, accelerate, etc.). "
            "Ask about the outcome directly - e.g. 'Will loan nonperformance in Dallas decrease?' - "
            "NOT 'Will the next Beige Book report that...'. "
            "Do NOT use explicit dates, months, or years in the question or resolution criteria. "
            "Resolution criteria describe WHAT to look for (conditions for Yes/No/Undetermined), "
            "never WHICH document - the pipeline always provides the correct document. "
            "Never reference specific report dates, release dates, or months/years. "
            "Focus on district-specific topics (Dallas, St. Louis, Boston, etc.) and metrics "
            "that the Beige Book explicitly reports on. "
            "Resolution criteria MUST state: resolve Yes/No ONLY when the document explicitly "
            "addresses the topic in the relevant district section; if the topic is not reported "
            "or not mentioned, resolve as Undetermined (unverifiable). "
            "For increase/decrease/improvement questions: criteria MUST explicitly state that "
            "flat, stable, unchanged, held steady, 'about flat', or no change = No. "
            "Generate only questions that are likely to be explicitly addressed in the relevant "
            "district section - avoid topics that may be unreported."
        ),
        examples=[
            "Will loan nonperformance in the Dallas district decrease?",
            "Will employment growth in the Philadelphia district slow?",
            "Will manufacturing activity in the St. Louis district improve?",
        ],
        bad_examples=[
            "Will the next Beige Book report that loan nonperformance in Dallas has decreased? # REASON: uses report framing",
            "Will the Federal Reserve Bank of Dallas's October 2024 Beige Book report that X? # REASON: references specific dates",
            "Will economic activity in the Cleveland district change? # REASON: too vague (increase or decrease?)",
            "Will niche industry X in district Y improve? # REASON: unlikely to be explicitly addressed",
        ],
    ),
    labeler=FileSetDocumentLabeler(
        file_set_id=FILESET_ID,
        temporal_constraint=TemporalConstraint.NEXT_DOCUMENT,
        answer_type=answer_type,
        confidence_threshold=0.7,
    ),
    context_generators=[
        QdrantContextGenerator(
            file_set_id=FILESET_ID,
            temporal_direction="before",
        )
    ],
    renderer=QuestionRenderer(
        answer_type=answer_type,
        template=template,
    ),
)

## 4. Run pipeline

Run `lr.transforms.run` to execute the pipeline and produce a dataset. This step can take time because generation, labeling, and context retrieval all happen here.

In [5]:
dataset = lr.transforms.run(
    pipeline,
    max_questions=3000,
    name="Beige Book SDK",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           e4f47a80-745c-427d-8c68-b0a75b45558a                                                       │
│                                                                                                                 │
│    Total cost: $32.64                                                                                           │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step              ┃ Progress             ┃   In ┃  Out ┃ Rejected ┃ Errors ┃ Rejection Reasons ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGener… │ Complete             │   26 │  822 │        0 │      0 │ -                 │       8s │  │
│  │ ForwardLookingQu… │ Complete             │  822 │ 4025 │       85 │      0 │ date_close not    │   1m 31s │  │
│  │                   │                      │      │      │          │        │ after event_date  │          │  │
│  │                   │                      │      │      │          │        │ (85)              │          │  │
│  │ FileSetDocumentL… │ Complete             │ 4025 │ 2827 │     1198 │      0 │ Undetermined      │   7m 26s │  │
│  │                   │                      │      │      │          │        │ label (497)       │          │  │
│  │                   │                      │      │      │          │        │ unknown (490)     │          │  │
│  │                   │                      │      │      │          │        │ +5 more           │          │  │
│  │ QdrantContextGen… │ Complete             │ 2827 │ 2827 │        0 │      0 │ -                 │    6m 5s │  │
│  │ QuestionRenderer… │ Complete             │ 2827 │ 2827 │        0 │      0 │ -                 │       0s │  │
│  └───────────────────┴──────────────────────┴──────┴──────┴──────────┴────────┴───────────────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=6418175;https://dashboard.lightningrod.ai/?redirect=/datasets/63c4444c-7fa7-4565-86c5-9ca293b13cf5\https://dashboard.lightningrod.ai/?redirect=/datasets/63c4444c-7fa7-4565-86c5-9ca293b13cf5]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 63c4444c-7fa7-4565-86c5-9ca293b13cf5
Rows: 3860


## 5. Prepare for training

Clean and split the dataset with `prepare_for_training`. It removes low-quality/invalid rows, deduplicates repeated questions, and creates train/test splits for fair evaluation.

In [ ]:
from lightningrod.training import prepare_for_training, SplitParams, DedupParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    split=SplitParams(test_size=0.15),
)
print(f"train: {train_dataset.num_rows}, test: {test_dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 3860 samples                                                                                   │
│                                                                                                                 │
│    Filter:  Dropped 1033 invalid → 2827 remain                                                                  │
│    Dedup:   Removed 1112 duplicates (2827 → 1715)                                                               │
│      ('Will manufacturing activity in the Chicago district increas..., None): 30 samples → 1                    │
│      ('Will retail sales in the Dallas district increase?', None): 24 samples → 1                               │
│      ('Will consumer spending in the Minneapolis district increase..., None): 22 samples → 1                    │
│    Split:   Splits: 1333 train | 258 test (0 dropped, no prediction_date)                                       │
│             124 train samples removed for leakage                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

train: 1333, test: 258


## 6. Train with SDK (GRPO/RL)

Set training hyperparameters with `GRPOTrainingConfig`, then launch a fine-tuning job on the training split via `lr.training.create`.

In [ ]:
from lightningrod import GRPOTrainingConfig, training

training_config = GRPOTrainingConfig(
        base_model_id="openai/gpt-oss-120b",
        training_steps=40,
        batch_size=32,
        max_response_length=16384,
        lora_rank=32,
        num_rollouts=4,
        learning_rate=4e-5,
)

training_job = lr.training.create(
    training_config,
    dataset=train_dataset,
    name="beige-book-sdk",
)
print(f"Job: {training_job.id}  status: {training_job.status}")

RemoteProtocolError: Server disconnected without sending a response.

## 7. Evaluate on test split

Evaluate on held-out data with `lr.evals.run`, then inspect `training.print_eval` output to compare the fine-tuned model against the base model.

In [ ]:
eval_job = lr.evals.run(
    training_config,
    training_job,
    test_dataset,
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Job ID: 8549d7d0-0bb2-43d0-bcdf-1447eb3106a3                                                                 │
│    Dataset: 35011ea3-f6bb-4af5-9efd-437858d7ce24                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓                                                                 │
│  ┃ Metric              ┃    Base ┃ Fine-tuned ┃                                                                 │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩                                                                 │
│  │ brier_score         │  0.3013 │     0.2166 │                                                                 │
│  │ ece                 │  0.2675 │     0.0792 │                                                                 │
│  │ mc_ece              │       — │          — │                                                                 │
│  │ mean_reward         │ -0.3013 │    -0.2166 │                                                                 │
│  │ mean_valid_reward   │ -0.3013 │    -0.2166 │                                                                 │
│  │ n_samples           │      96 │         96 │                                                                 │
│  │ n_valid             │      96 │         96 │                                                                 │
│  │ parse_rate          │  1.0000 │     1.0000 │                                                                 │
│  │ total_cost          │       — │     0.0620 │                                                                 │
│  │ total_input_tokens  │  166192 │     164040 │                                                                 │
│  │ total_output_tokens │   44873 │      73836 │                                                                 │
│  └─────────────────────┴─────────┴────────────┘                                                                 │
│                                                                                                                 │
│    Cost:  $0.06                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯